Nome: Hugo Alves Cardoso

e-mail: hac3@cesar.school

dataset: https://www.kaggle.com/datasets/mloey1/ahcd1

In [1]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import TensorDataset
import torch.optim as optim
import torch.nn.functional as F

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
path_test_image = "csvTestImages 3360x1024.csv"

# Load the latest version
df_test_image = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mloey1/ahcd1",
  path_test_image,
)

/tmp/ipykernel_12756/2934250085.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_test_image = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ahcd1' dataset.


In [3]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
path_test_label = "csvTestLabel 3360x1.csv"

# Load the latest version
df_test_label = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mloey1/ahcd1",
  path_test_label,
)

/tmp/ipykernel_12756/3828742193.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_test_label = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ahcd1' dataset.


In [4]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
path_train_images = "csvTrainImages 13440x1024.csv"

# Load the latest version
df_train_images = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mloey1/ahcd1",
  path_train_images,
)

/tmp/ipykernel_12756/673546917.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_train_images = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ahcd1' dataset.


In [5]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
path_train_labels = "csvTrainLabel 13440x1.csv"

# Load the latest version
df_train_label = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mloey1/ahcd1",
  path_train_labels,
)

/tmp/ipykernel_12756/1909517104.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_train_label = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ahcd1' dataset.


In [6]:
# Convert the dataframes of images and labels into DataLoader for training the model

X_tensor = torch.tensor(df_train_images.values, dtype=torch.float32) / 255.0
y_tensor = torch.tensor(df_train_label.values, dtype=torch.long) - 1

if y_tensor.ndim > 1 and y_tensor.shape[1] == 1:
    y_tensor = y_tensor.squeeze()

train_dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

X_tensor_test = torch.tensor(df_test_image.values, dtype=torch.float32) / 255.0
y_tensor_test = torch.tensor(df_test_label.values, dtype=torch.long) - 1

if y_tensor_test.ndim > 1 and y_tensor_test.shape[1] == 1:
    y_tensor_test = y_tensor_test.squeeze()

test_dataset = TensorDataset(X_tensor_test, y_tensor_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
class MeuModelo(nn.Module):
  def __init__(self):
    super(MeuModelo, self).__init__()
    self.linear1 = nn.Linear(1024, 1024)
    self.relu1 = nn.ReLU()
    self.dropout1 = nn.Dropout(0.2) # Added dropout
    self.linear2 = nn.Linear(1024, 1024)
    self.relu2 = nn.ReLU()
    self.dropout2 = nn.Dropout(0.2) # Added dropout
    self.linear3 = nn.Linear(1024, 1024)
    self.relu3 = nn.ReLU()
    self.dropout3 = nn.Dropout(0.2) # Added dropout
    self.linear4 = nn.Linear(1024, 1024)
    self.relu4 = nn.ReLU()
    self.dropout4 = nn.Dropout(0.2) # Added dropout
    self.linear5 = nn.Linear(1024, 1024)
    self.relu5 = nn.ReLU()
    self.dropout5 = nn.Dropout(0.2) # Added dropout
    self.linear6 = nn.Linear(1024, 1024)
    self.relu6 = nn.ReLU()
    self.dropout6 = nn.Dropout(0.2) # Added dropout
    self.linear7 = nn.Linear(1024, 28)

  def forward(self, x):
    x = self.dropout1(self.relu1(self.linear1(x)))
    x = self.dropout2(self.relu2(self.linear2(x)))
    x = self.dropout3(self.relu3(self.linear3(x)))
    x = self.dropout4(self.relu4(self.linear4(x)))
    x = self.dropout5(self.relu5(self.linear5(x)))
    x = self.dropout6(self.relu6(self.linear6(x)))
    x = self.linear7(x)
    output = torch.log_softmax(x, dim=1)
    return output

model = MeuModelo()

In [8]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [9]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [10]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")
device

device(type='cpu')

In [11]:
optimizer = optim.Adadelta(model.parameters(), lr=1)
epochs = 24

scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

In [12]:
for epoch in range(1, epochs + 1):
  train(10, False, model, device, train_loader, optimizer, epoch)
  test(model, device, test_loader)
  scheduler.step()

torch.save(model.state_dict(), "model.pt")

Train Epoch: 1 [0/13439 (0%)]	Loss: 3.336404
Train Epoch: 1 [320/13439 (2%)]	Loss: 3.327469
Train Epoch: 1 [640/13439 (5%)]	Loss: 3.324032
Train Epoch: 1 [960/13439 (7%)]	Loss: 3.321185
Train Epoch: 1 [1280/13439 (10%)]	Loss: 3.333866
Train Epoch: 1 [1600/13439 (12%)]	Loss: 3.330335
Train Epoch: 1 [1920/13439 (14%)]	Loss: 3.313909
Train Epoch: 1 [2240/13439 (17%)]	Loss: 3.354946
Train Epoch: 1 [2560/13439 (19%)]	Loss: 3.335256
Train Epoch: 1 [2880/13439 (21%)]	Loss: 3.326985
Train Epoch: 1 [3200/13439 (24%)]	Loss: 3.331943
Train Epoch: 1 [3520/13439 (26%)]	Loss: 3.337399
Train Epoch: 1 [3840/13439 (29%)]	Loss: 3.328054
Train Epoch: 1 [4160/13439 (31%)]	Loss: 3.328146
Train Epoch: 1 [4480/13439 (33%)]	Loss: 3.329866
Train Epoch: 1 [4800/13439 (36%)]	Loss: 3.338588
Train Epoch: 1 [5120/13439 (38%)]	Loss: 3.341384
Train Epoch: 1 [5440/13439 (40%)]	Loss: 3.333343
Train Epoch: 1 [5760/13439 (43%)]	Loss: 3.325783
Train Epoch: 1 [6080/13439 (45%)]	Loss: 3.328717
Train Epoch: 1 [6400/13439 (48

Durante a realização do treinamento do modelo, uma grande dificuldade durante o desenvolvimento e treinamento foi encontrar uma configuração do modelo (quantidade de camadas e quantidade de neurônio de camadas, quantidade de dropout entre camadas), configuração dos parâmetros de treinamento (learning rate e épocas) e decaimento dos parâmetros.

Em alguns casos, tentou-se realizar o treinamento utilizando um modelo com menos camadas internas, com uma menor quantidade de neurônios por camadas, nesses casos se observou que o modelo passava muitas épocas sem obter uma melhoras nos valores de Loss, tanto nos datasets de treinamento quanto no dataset de validação, indicando um cenário de underfitting. Dessa forma foi aumentado o numero de camadas e neurônios no modelo.

Com a nova configuração, foi observado que o modelo demorava mais tempo para realizar o treinamento, e também, em certos casos, obtinha melhoras somente para o dataset de treinamento, mantendo a mesma performance ou piorando a performance para o dataset de validação, indicando um possível cenário de overfitting.

Finalmente foi encontrada uma configuração que foi possível obter um valor de acurácia para o dataset de validação de 70%, variando constantemente os parâmetros mencionados na tentativa de fazer com que o modelo convergisse para o valor de validação de pelo menos 70%

In [14]:
class MeuModelo2(nn.Module):
  def __init__(self):
    super(MeuModelo2, self).__init__()
    self.linear1 = nn.Linear(1024, 1024)
    self.relu1 = nn.ReLU()
    self.dropout1 = nn.Dropout(0.1) # Added dropout
    self.linear2 = nn.Linear(1024, 1024)
    self.relu2 = nn.ReLU()
    self.dropout2 = nn.Dropout(0.1) # Added dropout
    self.linear3 = nn.Linear(1024, 1024)
    self.relu3 = nn.ReLU()
    self.dropout3 = nn.Dropout(0.1) # Added dropout
    self.linear4 = nn.Linear(1024, 1024)
    self.relu4 = nn.ReLU()
    self.dropout4 = nn.Dropout(0.1) # Added dropout
    self.linear5 = nn.Linear(1024, 1024)
    self.relu5 = nn.ReLU()
    self.dropout5 = nn.Dropout(0.1) # Added dropout
    self.linear6 = nn.Linear(1024, 1024)
    self.relu6 = nn.ReLU()
    self.dropout6 = nn.Dropout(0.1) # Added dropout
    self.linear7 = nn.Linear(1024, 28)

  def forward(self, x):
    x = self.dropout1(self.relu1(self.linear1(x)))
    x = self.dropout2(self.relu2(self.linear2(x)))
    x = self.dropout3(self.relu3(self.linear3(x)))
    x = self.dropout4(self.relu4(self.linear4(x)))
    x = self.dropout5(self.relu5(self.linear5(x)))
    x = self.dropout6(self.relu6(self.linear6(x)))
    x = self.linear7(x)
    output = torch.log_softmax(x, dim=1)
    return output

model2 = MeuModelo2()

In [15]:
optimizer = optim.Adadelta(model2.parameters(), lr=1)
epochs = 24

#scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

In [16]:
for epoch in range(1, epochs + 1):
  train(10, False, model2, device, train_loader, optimizer, epoch)
  test(model2, device, test_loader)
  scheduler.step()

torch.save(model2.state_dict(), "model2.pt")

Train Epoch: 1 [0/13439 (0%)]	Loss: 3.331043
Train Epoch: 1 [320/13439 (2%)]	Loss: 3.333006
Train Epoch: 1 [640/13439 (5%)]	Loss: 3.340962
Train Epoch: 1 [960/13439 (7%)]	Loss: 3.333038
Train Epoch: 1 [1280/13439 (10%)]	Loss: 3.335451
Train Epoch: 1 [1600/13439 (12%)]	Loss: 3.350874
Train Epoch: 1 [1920/13439 (14%)]	Loss: 3.340505
Train Epoch: 1 [2240/13439 (17%)]	Loss: 3.333653
Train Epoch: 1 [2560/13439 (19%)]	Loss: 3.335161
Train Epoch: 1 [2880/13439 (21%)]	Loss: 3.334034
Train Epoch: 1 [3200/13439 (24%)]	Loss: 3.332791
Train Epoch: 1 [3520/13439 (26%)]	Loss: 3.332709
Train Epoch: 1 [3840/13439 (29%)]	Loss: 3.336139
Train Epoch: 1 [4160/13439 (31%)]	Loss: 3.346500
Train Epoch: 1 [4480/13439 (33%)]	Loss: 3.327934
Train Epoch: 1 [4800/13439 (36%)]	Loss: 3.326606
Train Epoch: 1 [5120/13439 (38%)]	Loss: 3.333019
Train Epoch: 1 [5440/13439 (40%)]	Loss: 3.339649
Train Epoch: 1 [5760/13439 (43%)]	Loss: 3.325227
Train Epoch: 1 [6080/13439 (45%)]	Loss: 3.256929
Train Epoch: 1 [6400/13439 (48

Diminuindo a quantidade de dropout de neurônios no modelo, e não realizando a diminuição do learning rate com o scheduler, o modelo foi capaz de obter acuracia de 81%, melhora significativa aos 70% anteriores